# Ноутбук 4 — дообучение модели через LoRA

Дообучаю `deepvk/llava-gemma-2b-lora` на подвыборке `LLaVA-Instruct-ru`.
Модель гружу в fp16 (3B спокойно влезает в T4), обучаю только LoRA-адаптеры на языковой части,
vision-башню морозим. Чекпоинты — на Google Drive (чтобы пережить обрыв сессии).

**Порядок работы:** сначала прогнать всё с `SUBSET_N = 100` (проверка, что пайплайн живой),
потом поставить `SUBSET_N = 2000` и перезапустить ячейки с данными и обучением.

Важно: у `LLaVA-Instruct-ru` в поле image лежит путь на COCO, а не картинка — поэтому
нужные картинки докачиваю отдельно с images.cocodataset.org.

## Библиотеки

Тот же набор версий, что и раньше (модель заводится только на нём), плюс peft и bitsandbytes.
**После установки перезапусти сессию.**

In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "huggingface_hub==0.24.6" "accelerate==0.33.0" "datasets==2.21.0" "peft==0.12.0" "bitsandbytes==0.43.3" "pillow<12"

После перезапуска — проверка версий.

In [ ]:
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print('GPU:', torch.cuda.is_available())

## Логин в HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Подключаю Google Drive

Сюда пойдут чекпоинты и итоговый адаптер. Colab попросит разрешить доступ.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/vlm_project'
CKPT_DIR = DRIVE_DIR + '/checkpoints'
ADAPTER_DIR = DRIVE_DIR + '/lora_adapter'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
print('чекпоинты ->', CKPT_DIR)

## Настройки и подвыборка данных

Начни со 100, потом поставь 2000.

In [ ]:
SUBSET_N = 100
LORA_R = 16
LORA_ALPHA = 32
LR = 2e-4
EPOCHS = 1
GRAD_ACCUM = 16

from datasets import load_dataset
full_ds = load_dataset('deepvk/LLaVA-Instruct-ru', split='train')
sub = full_ds.shuffle(seed=42).select(range(SUBSET_N))
print('взял примеров:', len(sub))

## Качаю нужные картинки COCO

Только те, что попали в подвыборку. Кладу в /content (быстро). Если сессия оборвётся —
докачаются заново за пару минут.

In [ ]:
import requests
from PIL import Image
from tqdm.auto import tqdm

COCO_DIR = '/content/coco_images'
os.makedirs(COCO_DIR, exist_ok=True)

def coco_fname(path):
    return path.replace('\\', '/').split('/')[-1]

def coco_local(path):
    return os.path.join(COCO_DIR, coco_fname(path))

def download_coco(path):
    local = coco_local(path)
    if os.path.exists(local):
        return True
    url = 'http://images.cocodataset.org/train2017/' + coco_fname(path)
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(local, 'wb') as f:
            f.write(r.content)
        return True
    except Exception:
        return False

ok_idx = [i for i, ex in enumerate(tqdm(sub)) if download_coco(ex['image'])]
train_ds = sub.select(ok_idx)
print('годных примеров с картинками:', len(train_ds), 'из', len(sub))

## Гружу модель в fp16 и навешиваю LoRA

Модель 3B в fp16 занимает ~5.6 ГБ, в T4 влезает с запасом — 4-bit квантизация не нужна
(к тому же на этом CUDA bitsandbytes её не тянет, а vision в 4-bit ломает картинки).
Учим только LoRA на языковой части: vision-башню и проектор морозим.
LoRA-параметры держу в fp32 для стабильности обучения.

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor, AutoTokenizer

model_id = 'deepvk/llava-gemma-2b-lora'
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map={'': 0}
)
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print('модель в fp16')

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)

for n, p in model.named_parameters():
    if 'vision_tower' in n or 'multi_modal_projector' in n:
        p.requires_grad_(False)

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.print_trainable_parameters()

## Сборка обучающего примера

Из диалога беру первый вопрос человека и первый ответ ассистента. Учу модель только на ответе
(вопрос и картинку маскирую значением -100). Батч = 1, картинки разного размера друг другу не мешают.

In [ ]:
def first_qa(conv):
    human = None
    gpt = None
    for c in conv:
        if c['from'] == 'human' and human is None:
            human = c['value']
        elif c['from'] == 'gpt' and human is not None:
            gpt = c['value']
            break
    return human, gpt

def collate(batch):
    ex = batch[0]
    human, gpt = first_qa(ex['conversations'])
    if not human:
        human = '<image>\nОпиши картинку.'
    if '<image>' not in human:
        human = '<image>\n' + human
    if not gpt:
        gpt = '.'
    user = {'role': 'user', 'content': human}
    asst = {'role': 'assistant', 'content': gpt}
    prompt_text = tokenizer.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template([user, asst], tokenize=False, add_generation_prompt=False)
    img = Image.open(coco_local(ex['image'])).convert('RGB')
    enc = processor(images=[img], text=full_text, return_tensors='pt')
    prompt_len = processor(images=[img], text=prompt_text, return_tensors='pt')['input_ids'].shape[1]
    labels = enc['input_ids'].clone()
    labels[:, :prompt_len] = -100
    enc['labels'] = labels
    enc['pixel_values'] = enc['pixel_values'].to(torch.float16)
    return dict(enc)

## Проверка на одном примере

Прогоняю один пример через модель — если посчитался конечный loss, весь пайплайн
(картинка -> токены -> маска -> forward -> loss) рабочий.

In [ ]:
one = collate([train_ds[0]])
one = {k: v.to(model.device) for k, v in one.items()}
model.train()
out = model(**one)
print('loss на одном примере:', float(out.loss))
del one, out
torch.cuda.empty_cache()

## Обучение

Если в папке чекпоинтов уже что-то есть (после обрыва) — обучение само продолжится с последнего.

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)

has_ckpt = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
trainer.train(resume_from_checkpoint=has_ckpt)

## Сохраняю LoRA-адаптер

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('адаптер сохранён:', ADAPTER_DIR)
print('содержимое:', os.listdir(ADAPTER_DIR))

## Что дальше

1. Если мини-прогон (100) отработал и loss адекватный — поставь `SUBSET_N = 2000`
   и перезапусти ячейки начиная с «Настройки и подвыборка данных».
2. После полного обучения адаптер лежит на Drive (`lora_adapter`).
3. В ноутбуке 5 (оценка после обучения) грузим базовую модель в fp16, поверх — этот адаптер,
   и прогоняем те же GQA/MMBench с тем же N и seed. Сравниваем с базлайном.